# 07b · 新数据：MaveDB

## 这一章在整个项目里的位置

```
00–06 ✅ → [07 特征和基线] → 08 深度学习 → 09 全蛋白组预测 → 10 对外 API
              │
              ├─ 07a 设计：特征清单、考试规则 ✅
              ├─ 07b 新数据：MaveDB 进 raw 和 bronze   ← 你在这里
              ├─ 07c 标签：两把刻度，同一把尺子
              ├─ 07d 结构特征和图：AlphaFold
              ├─ 07e 语言模型特征：ESM-2
              └─ 07f 特征表 + 第一个基线模型 + MLflow
```

## 背景：为什么需要这一章

07a 定标签的时候碰到一个难题：每个 DMS 实验的分数尺子都不一样，要换算到同一把尺子上，得知道每个实验里"像野生型"是多少分、"完全失活"是多少分。

查资料后发现，标准做法（Gray 2018、Lafita 2024）是用实验里**自带的两把刻度**：
- **同义突变**：DNA 变了，氨基酸没变，所以和野生型一样
- **无义突变**：提前出现终止密码子，蛋白被截断，所以失活

问题是 ProteinGym 加工数据时，**把这两种突变都删掉了**，只留了错义突变。原始数据在哪里？在 **MaveDB**，一个专门存放 DMS 这类实验数据的公共数据库。

查的时候还回答了你问的"数据够不够"：MaveDB 有 2,777 组实验，其中人类的 1,200 组。

## 你在哪里操作

```
你：在这个 notebook 里运行一格
   ├─ 第 2 步：送货员 Lambda 把 MaveDB 的整库打包搬进 S3 raw（03b 那一套）
   └─ 第 4 步：Databricks 的流水线把它装进 bronze（04 那一套）
看结果：这里的输出；Databricks 网页的 Catalog → qafi_dev → bronze
```

## 这一章做的 5 件事

| # | 做什么 | 比喻 |
|---|---|---|
| 1 | 认识 MaveDB | 原始实验记录本 vs 整理好的习题集 |
| 2 | 进 raw | 进货，和 03b 一样 |
| 3 | 包里有什么 | 拆箱验货 |
| 4 | 装进 bronze | 原样上架，和 04 一样 |
| 5 | 看一眼 | 盘点库存 |

**花费**：S3 多存 2 GB，每月约 0.05 美元；其他都是 0。**时间**：大约 15 分钟，其中 6 分钟是等 Databricks。

---

## 准备

运行下面这一格：连上 Databricks 和 AWS。**不用看代码。**

In [ ]:
import os, subprocess
from pathlib import Path
import boto3
from databricks.sdk import WorkspaceClient
from qafi_platform import ingest, lakehouse
from qafi_platform.lakehouse import sql
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
w = WorkspaceClient(profile="qafi")
aws = boto3.Session(profile_name="qafi", region_name="eu-central-1")
CATALOG, JOB, RAW = "qafi_dev", "qafi training data", "qafi-186972156153-dev-raw"
SOURCES = ROOT / "config" / "sources.toml"
print("Databricks:", w.config.host)
print("AWS:", aws.client("sts").get_caller_identity()["Arn"])

> 如果 AWS 报错（`session has expired`）：在终端运行 `aws login --profile qafi`，再运行这一格。

---

## 第 1 步 · 认识 MaveDB

### 在干什么
弄清楚 MaveDB 是什么，为什么用它，以及为什么下载"整库打包"。

### 背景
比喻：
- **ProteinGym** 像一本**整理好的习题集**：从各个实验里挑出错义突变，统一了方向（分数越高越好），方便大家比赛打分
- **MaveDB** 像**原始实验记录本**：作者上传的原样数据。同义、无义、错义突变都在，还附带了注释

我们要的两把刻度，只在原始记录本里。

### 用了什么技术，为什么
- **MaveDB**（mavedb.org）：多重变异效应实验（MAVE，DMS 就是其中一种）的公共数据库，有公开接口
- **整库打包**（Zenodo 上的 "MaveDB Bulk Download"）：所有公开数据一个文件，1.91 GB，2026 年 6 月 25 日版
- 为什么用整库打包，不一个个调接口：**一个文件、一个版本号**，以后任何人用同一个版本都能得到一模一样的数据。接口每天都在变，今天查和明天查结果可能不同
- **授权**：CC0，等于放弃一切权利，随便用

**和你的研究有关的两点**：
- 旧论文用来考试的 **Domainome**（Beltran 2025，500 多个人类蛋白结构域），在 MaveDB 里有 503 组
- 专门用 DMS 数据**微调 ESM** 的论文（Lafita 2024）用的也是 MaveDB，标签用的就是这两把刻度

### 面试怎么说
> "ProteinGym is a curated benchmark that keeps only missense variants, so the anchors needed to put assays on one scale were gone. I went back to MaveDB, the primary repository, and ingested its versioned bulk dump, which still has synonymous and nonsense variants."

---

## 第 2 步 · 进 raw

### 在干什么
把 MaveDB 写进数据源清单，让送货员 Lambda 从 Zenodo 直接搬进 S3 raw。

### 背景
和 03b 进 ClinVar、UniProt 完全一样：清单里写上名字、版本、网址；送货员分块下载，验货 Lambda 检查、开收货单。文件不经过你的电脑。

### 用了什么技术，为什么
[`config/sources.toml`](../../config/sources.toml) 里新加了一条，`refresh = "manual"`：MaveDB 每隔几个月出一次新打包，换版本会改变训练数据，所以要人来改版本号、提交审核。

### 执行

In [ ]:
mavedb = [s for s in ingest.load_sources(SOURCES) if s["dataset"] == "mavedb"]
ingest.fetch_sources(aws.client("lambda"), aws.client("s3"), RAW, mavedb, timeout=900)

### 结果怎么看
- 如果显示 `skipped ... (already in raw)`：说明它已经在 raw 里了（我写这一章时试跑过），送货员不会重复搬。这就是 03b 讲过的"重跑安全"
- 如果显示 `accepted ... 1.91 GB`：刚刚搬完，验货通过

### 面试怎么说
> "New sources are one entry in a config file; a Lambda copies the publisher's file straight into the raw bucket with ranged requests, and a second Lambda validates it and writes a receipt. Re-running skips what is already there."

---

## 第 3 步 · 包里有什么

### 在干什么
拆箱验货：看看打包里有哪些文件，我们用哪些。

### 背景
| 文件 | 内容 | 用不用 |
|---|---|---|
| `main.json` | 所有实验的说明：目标蛋白、物种、文献、分数含义 | 用 |
| `csv/{编号}.scores.csv` | 每组实验一个表，每个突变一行，分数在 `scores.score` | 用 |
| `csv/{编号}.annotations.csv` | MaveDB 把突变对到人类参考序列上的结果，加上 **VEP 注释**：每个突变是同义、无义还是错义 | 用 |
| `counts.csv`、`mapped/`、`va/` | 测序读数、更详细的映射格式 | 现在不用 |

**原始数据总有意外**：这个包是在 Mac 上打的，每个 CSV 旁边都多了一个 `._` 开头的"影子文件"，一共 3,947 个，里面是 Mac 的属性信息，不是数据。Mac 自带的 `tar` 命令列文件时会把它们藏起来，Python 却能看到。第一次解析时就是被它们弄出了编码错误。代码现在会跳过它们，单元测试里也专门放了一个这种文件。

### 用了什么技术，为什么
解析代码在 [`databricks/src/lakehouse/mavedb.py`](../../databricks/src/lakehouse/mavedb.py)：
- **边读边处理**（流式）：1.91 GB 的压缩包从头读到尾只读一遍，不解压到硬盘
- **原样保存**：所有值都存成文字。每组实验自己定义的额外分数列，合成一个 JSON 字段，一个都不丢
- 先写成 parquet 小文件，再由 Spark 装成 Delta 表。和 04 号处理旧全蛋白组预测是同一个办法

### 面试怎么说
> "The archive is streamed once into parquet parts and loaded as Delta, with every value kept as delivered. The first run failed on thousands of macOS '._' metadata files that the system tar hides; the parser now skips them and a unit test pins it."

---

## 第 4 步 · 装进 bronze

### 在干什么
Databricks 的流水线多了一步 `bronze_mavedb`，把打包装成三张 bronze 表。

### 背景
流水线现在是这样：

```
copy_raw ─┬─▶ bronze_reference ─┬─▶ silver_dms ─▶ quality_checks ─▶ gold
          ├─▶ bronze_dms ────────┘
          └─▶ bronze_mavedb（新）
```

这次只需要跑**其中两步**：`copy_raw`（把文件从 S3 搬进 Databricks 的文件区）和 `bronze_mavedb`。用 `--only` 指定要跑的步骤，其他步骤这次跳过。这和 04 号用过的 Repair run 是同一个思路：只做需要做的。

### 执行
部署新图纸，再放一份提货单（04 号讲过：免费版的 Databricks 读不到我们的 S3，要靠临时下载链接）：

In [ ]:
run("cd databricks && databricks bundle deploy")
keys = [s["key"] for s in ingest.load_sources(SOURCES) if s["dataset"] in {"clinvar", "uniprot-human", "proteingym", "mavedb"}]
lakehouse.upload_transfer_list(w, CATALOG, lakehouse.transfer_links(aws.client("s3"), RAW, keys, hours=6))

### 结果怎么看
- `Deployment complete!`
- 最后一行是提货单的位置：`/Volumes/qafi_dev/raw/files/_transfer/links.json`

跑那两步，大约 5 分钟：

In [ ]:
run("cd databricks && databricks bundle run training_data --only copy_raw,bronze_mavedb")
lakehouse.latest_run(w, JOB)

### 结果怎么看
- `copy_raw` 和 `bronze_mavedb` 是 `SUCCESS`；`bronze_mavedb` 大约 4 分钟
- 其他步骤是 `SKIPPED`：这次没让它们跑

### 去网页上看
Databricks → **Catalog** → `qafi_dev` → `bronze`：多了三张表 `mavedb_score_sets`、`mavedb_scores`、`mavedb_annotations`，每张表都有说明。

### 面试怎么说
> "A new task in the same Lakeflow job loads MaveDB into bronze; running only the tasks that changed keeps iterations short."

---

## 第 5 步 · 看一眼

### 在干什么
盘点库存：三张表各有多少行，人类的有多少，Domainome 占多少。

In [ ]:
sql(w, f'''
SELECT 'mavedb_score_sets' AS table_name, count(*) AS rows FROM {CATALOG}.bronze.mavedb_score_sets
UNION ALL SELECT 'mavedb_scores', count(*) FROM {CATALOG}.bronze.mavedb_scores
UNION ALL SELECT 'mavedb_annotations', count(*) FROM {CATALOG}.bronze.mavedb_annotations''')

### 结果怎么看
- **2,777** 组实验，**820 万**个分数，**321 万**条注释

人类的、当前版本（没有被新版本取代）的蛋白实验：

In [ ]:
sql(w, f'''
SELECT CASE WHEN title ILIKE '%domainome%' THEN 'Domainome' ELSE '其他人类实验' END AS source,
       count(*) AS score_sets, count(DISTINCT target_uniprot) AS proteins, sum(CAST(num_variants AS BIGINT)) AS variants
FROM {CATALOG}.bronze.mavedb_score_sets
WHERE target_organism = 'Homo sapiens' AND target_category = 'protein_coding' AND superseded_by IS NULL
GROUP BY 1 ORDER BY 1''')

### 结果怎么看
- **Domainome**：503 组、388 个蛋白
- **其他人类实验**：616 组，只有 74 个蛋白，因为一个蛋白常常做了很多组（TP53 就有好几组）

也就是说，**能大幅增加训练蛋白的，主要是 Domainome**。我们的做法：
- 把 Domainome 当普通数据，按 07a 同样的规则按蛋白分边：约 80% 进训练，约 20% 进考卷
- 旧论文在这些考卷结构域上的预测我们都有，可以在**同一批题目**上和旧 QAFI 比
- Domainome 测的是蛋白丰度（稳定性），不完全等于"功能"。所以 07f 用 MLflow 做一个对照实验：训练时加不加 Domainome，在功能类的考题上哪个更好，用数据决定

VEP 注释里，同义、无义、错义各有多少：

In [ ]:
sql(w, f'''
SELECT a.vep_vep_functional_consequence AS consequence, count(*) AS variants
FROM {CATALOG}.bronze.mavedb_annotations a JOIN {CATALOG}.bronze.mavedb_score_sets s USING (urn)
WHERE s.target_organism = 'Homo sapiens' AND s.superseded_by IS NULL
GROUP BY 1 ORDER BY 2 DESC LIMIT 6''')

### 结果怎么看
- `missense_variant` 约 63 万，`stop_gained`（无义）约 3.2 万，`synonymous_variant`（同义）约 2.4 万：**两把刻度都在**
- `NA` 有 225 万：主要是多点突变，或者 MaveDB 还没映射的实验。07c 会结合 `hgvs_pro` 那一列一起判断

### 公司里什么情况怎么做
- 接入新数据源，先**原样装进 bronze**，再盘点，然后才决定怎么清洗、怎么用。不要一边接入一边清洗：出了问题分不清是数据的问题还是清洗的问题
- 盘点要回答"它能解决我的什么问题"：这里是"刻度在不在""能多出多少蛋白"

### 面试怎么说
> "MaveDB added 1,119 human score sets; outside Domainome they cover only 74 proteins, so the real gain in protein diversity is Domainome. I split it by protein like everything else, keep its held-out domains as the external benchmark against the old model, and will test whether its stability data helps function prediction."

---

## 提交代码

In [ ]:
run("git add -A")
run("uv run pre-commit run", check=False)   # 第一次会自动整理 notebook 的格式
run("git add -A")
run('(git diff --cached --quiet || git commit -q -m "feat: 07b MaveDB bulk download into raw and bronze") && git push -q')

推送后去 GitHub 的 Actions 页面看一眼，应该全绿。

---

## 本章小结

| 步骤 | 做了什么 | 为什么 |
|---|---|---|
| 1 MaveDB | 从"习题集"回到"原始记录本" | 同义、无义突变只在原始数据里 |
| 2 进 raw | 清单加一条，送货员搬进 S3 | 和其他数据源同一套流程，版本写死 |
| 3 拆箱 | 流式解析，原样保存；跳过 Mac 影子文件 | 1.9 GB 只读一遍；原始数据总有意外 |
| 4 bronze | 流水线多一步，只跑需要的两步 | 不重复做已经做过的事 |
| 5 盘点 | 人类 1,119 组；Domainome 占 388 个蛋白 | 弄清新数据能解决什么问题 |

**一句话**：标签需要的两把刻度、训练需要的更多蛋白，都从 MaveDB 这个原始数据库里找到了，并且已经原样装进了 bronze。

## 面试题

1. **Why go back to the primary repository instead of using the curated benchmark?**
   > "The benchmark had removed the synonymous and nonsense variants that anchor each assay's scale. Only the primary data could put assays with different readouts on one scale."

2. **Why a versioned bulk dump instead of the API?**
   > "One file with one version is reproducible: anyone loading the same version gets the same data. An API answers differently from day to day."

3. **What would you check when a new data source arrives?**
   > "Load it unchanged into bronze, then profile it against the question it should answer, here whether the anchors exist and how many new proteins it adds, before writing any cleaning logic."

**下一步 07c · 标签**：分出同义、无义、错义突变，对到 UniProt 编号，用两把刻度把所有实验换算到同一把尺子上。